# Chapter 9 — From Similarity to Search

**Book alignment:** Embeddings From First Principles, Chapter 9

**Question this notebook isolates:** Retrieval is `embed → score → sort → cut`. Which
parameters change the *answer* (representation, metric, `k`, threshold) vs only the *speed*
(exact vs ANN)? And when ANN error appears, is it random — or does it systematically drop
the hardest neighbours? (On RELATE's 1,173 vectors, HNSW reproduces the exact top-10 even
at `ef_search=10` — the committed Wave 1 measurement.)

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    sub = "wave1/artifacts/v02" if wave == "wave1-v02" else f"{wave}/artifacts"
    return json.loads((EXP / sub / name).read_text())

## 1. Retrieval in four lines — brute force, fully visible

In [ ]:
D = 32
corpus = rng.standard_normal((500, D))
corpus /= np.linalg.norm(corpus, axis=1, keepdims=True)

def retrieve(q, k=5):
    q = q / np.linalg.norm(q)
    scores = corpus @ q                      # one dot product per document
    order = np.argsort(-scores)              # sort, descending
    return order[:k], scores                 # take the top k

q = corpus[7] + rng.standard_normal(D) * 0.1
idx, scores = retrieve(q)
print("top-5 doc ids:", idx.tolist(), "  top score:", round(float(scores[idx[0]]), 3))
assert idx[0] == 7                           # nearest is the doc we perturbed
print("everything a vector DB adds is optimization around these four lines")

## 2. Answer-changing vs speed-changing parameters

In [ ]:
# k changes the answer (recall/precision tradeoff, and how much context a model sees):
assert len(retrieve(q, k=1)[0]) == 1 and len(retrieve(q, k=20)[0]) == 20

# a threshold changes the answer: without one you always return k, even when nothing is relevant
def retrieve_thresholded(q, k=5, tau=0.5):
    idx, sc = retrieve(q, k)
    return [i for i in idx if sc[i] >= tau]

junk = rng.standard_normal(D)
print("no threshold, junk query  :", len(retrieve(junk, k=5)[0]), "results returned")
print("with threshold 0.5        :", len(retrieve_thresholded(junk, tau=0.5)), "results returned")
assert len(retrieve_thresholded(junk, tau=0.5)) < 5
print("k and threshold are not display choices - they change what the system answers")

## 3. ANN error, where it appears, is not random — but RELATE is too small to show it

In [ ]:
# a toy IVF: cluster the corpus, search only the nearest `nprobe` clusters
from numpy.linalg import norm
K = 16
cent = corpus[rng.choice(len(corpus), K, replace=False)]
assign = np.argmin(((corpus[:, None] - cent[None]) ** 2).sum(-1), axis=1)

def ivf_retrieve(q, k=5, nprobe=1):
    qn = q / norm(q)
    near_clusters = np.argsort(-(cent @ qn))[:nprobe]
    cand = np.where(np.isin(assign, near_clusters))[0]
    return cand[np.argsort(-(corpus[cand] @ qn))][:k]

exact = set(retrieve(q, k=5)[0].tolist())
for nprobe in (1, 2, 4, 16):
    got = set(ivf_retrieve(q, k=5, nprobe=nprobe).tolist())
    print(f"nprobe={nprobe:2}  recall of exact top-5: {len(got & exact) / 5:.2f}")
assert len(set(ivf_retrieve(q, k=5, nprobe=1)) & exact) / 5 <= 1.0   # small nprobe can miss
print("ANN misses concentrate on the true neighbour that sat in an unsearched cluster")

In [ ]:
ann = art("wave1", "ann-vs-exact.json")
for ef, v in sorted(ann["ef_sweep"].items(), key=lambda kv: int(kv[0][2:])):
    print(f"  {ef:6} recall of exact top-10 = {v['recall_of_exact_top10']:.4f}"
          f"   recall of graded-relevant = {v['recall_of_graded_relevant']:.2f}")
assert ann["ef_sweep"]["ef10"]["recall_of_exact_top10"] > 0.99
assert all(v["recall_of_graded_relevant"] == 1.0 for v in ann["ef_sweep"].values())
print(f"\non {ann['n_items']} vectors, HNSW reproduces the exact top-10 even at ef_search=10")
print("the 'systematic ANN error' claim is real for MILLION-vector indexes; measure recall vs EXACT either way")

## What we earned

Retrieval is `embed → score → sort → cut`. `k` and the threshold change the answer; exact
vs ANN mostly changes the speed — and when ANN error appears it is systematic (it drops the
hardest neighbours), so you measure its recall against the *exact* result, never against
the labels. On a 1,173-vector index the index is solving a problem you may not have yet.

**Notebook 10 / Chapter 10** builds the cases where the nearest neighbour is fluent,
on-topic, geometrically closest — and wrong.